In [1]:
import pandas as pd
import numpy as np
import os
import io
import pickle
import warnings
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from contextlib import redirect_stdout

warnings.simplefilter(action='ignore', category=FutureWarning)

from ClassFunctions import precip_time_series, rainfall_analysis

# ---------------------------------------------------------------------------
# Config — change TEMP_RES to switch between modes:
#   5        -> native event detection from raw data
#   10/30/60 -> events inherited from the pre-existing 5-min pickle
# ---------------------------------------------------------------------------

BASE_DIR  = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/'
TEMP_RES  = 5     # <-- change this to 10, 30, or 60 for coarser resolutions
THRESHOLD = '11h'
N_WORKERS = 4

PICKLE_DIR = os.path.join(BASE_DIR, 'DanishRainDataPickles')
OUTPUT_DIR = os.path.join(BASE_DIR, 'DanishRainData_Outputs', f'{TEMP_RES}mins')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def get_directory(filename):
    """Determine input data directory from filename."""
    return 'DanishRainData_SVK' if 'svk' in filename else 'DanishRainData'


def get_pending_files():
    """
    Build the list of files that still need processing at this resolution.

    For 5-min: iterates raw CSV files directly.
    For coarser: iterates existing 5-min pickles (which define what's available).
    """
    if TEMP_RES == 5:
        # Collect all CSV files across both data directories
        files = []
        for directory in ['DanishRainData_SVK', 'DanishRainData']:
            input_dir = os.path.join(BASE_DIR, directory)
            if os.path.isdir(input_dir):
                files += [f for f in os.listdir(input_dir) if f.endswith('.csv')]
        pending = [
            f for f in files
            if not os.path.exists(os.path.join(OUTPUT_DIR, f'All_events_{f}'))
        ]
    else:
        # Use existing 5-min pickles as the source of truth
        pickle_files = os.listdir(PICKLE_DIR)
        pending = [
            p.replace('.pkl', '') for p in pickle_files
            if not os.path.exists(
                os.path.join(OUTPUT_DIR, f'All_events_{p.replace(".pkl", "")}'))
        ]

    return sorted(pending)

# ---------------------------------------------------------------------------
# Per-file processing
# ---------------------------------------------------------------------------

def process_file(filename):
    """
    Process one gauge file at TEMP_RES resolution.

    At 5-min: detects events natively from the raw CSV.
    At coarser resolutions: inherits events from the 5-min pickle.

    All internal print output is captured and returned as a log string
    so that parallel workers don't produce interleaved console output.

    Returns
    -------
    (filename, status, log)
    """
    directory   = get_directory(filename)
    input_path  = os.path.join(BASE_DIR, directory, filename)
    output_path = os.path.join(OUTPUT_DIR, f'All_events_{filename}')
    pickle_path = os.path.join(PICKLE_DIR, f'{filename}.pkl')

    if os.path.exists(output_path):
        return filename, 'skipped', ''

    buf = io.StringIO()
    try:
        with redirect_stdout(buf):

            # ----------------------------------------------------------
            # For coarser resolutions, check the 5-min pickle has events
            # before doing anything else
            # ----------------------------------------------------------
            if TEMP_RES != 5:
                if not os.path.exists(pickle_path):
                    return filename, 'no 5-min pickle found', buf.getvalue()
                with open(pickle_path, 'rb') as f:
                    five_min_ts = pickle.load(f)
                if not five_min_ts.events:
                    return filename, 'skipped (no 5-min events)', buf.getvalue()

            # ----------------------------------------------------------
            # Basic data check
            # ----------------------------------------------------------
            if not os.path.exists(input_path):
                return filename, f'error: input file not found ({input_path})', buf.getvalue()

            if pd.read_csv(input_path).empty:
                return filename, 'empty file', buf.getvalue()

            # ----------------------------------------------------------
            # Build time series
            # ----------------------------------------------------------
            reference_pickle = pickle_path if TEMP_RES != 5 else None

            ts = precip_time_series(
                input_path,
                temp_res=TEMP_RES,
                reference_pickle=reference_pickle
            )

            if ts.data['precipitation (mm/min)'].lt(0).any():
                return filename, 'negative values', buf.getvalue()

            ts.pad_and_resample()

            # Check for missing timesteps
            dt_index   = ts.data.index
            full_range = pd.date_range(
                start=dt_index.min(), end=dt_index.max(), freq=f'{TEMP_RES}T')
            missing = full_range.difference(dt_index)
            if len(missing) > 0:
                print(f"Warning: {len(missing)} missing timesteps after resampling")

            # ----------------------------------------------------------
            # Event detection / inheritance happens inside rainfall_analysis
            # ----------------------------------------------------------
            analysis = rainfall_analysis(THRESHOLD, ts)

            if not ts.events:
                return filename, 'no events found', buf.getvalue()

            # ----------------------------------------------------------
            # Save 5-min pickle (only in native mode)
            # ----------------------------------------------------------
            if TEMP_RES == 5:
                with open(pickle_path, 'wb') as f:
                    pickle.dump(ts, f, protocol=4)

            # ----------------------------------------------------------
            # Compute and save metrics
            # ----------------------------------------------------------
            analysis.get_metrics()
            df = pd.DataFrame(analysis.metrics)
            df['gauge_num']  = filename.split('_')[0]
            df['start_time'] = [e[0] for e in ts.events]
            df['end_time']   = [e[1] for e in ts.events]

            # Preserve link back to original 5-min event index
            if ts.original_event_indices is not None:
                df['event_num'] = ts.original_event_indices

            df.to_csv(output_path, index=False)
            return filename, f'success ({len(df)} events)', buf.getvalue()

    except Exception as e:
        return filename, f'error: {e}', buf.getvalue()

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

pending = get_pending_files()
mode    = 'native (5-min detection)' if TEMP_RES == 5 else f'inherited from 5-min pickle'
print(f"Resolution : {TEMP_RES} min  [{mode}]")
print(f"Files to process: {len(pending)}")

with Pool(processes=N_WORKERS) as pool:
    results = list(tqdm(
        pool.imap_unordered(process_file, pending),
        total=len(pending),
        desc='Processing'
    ))

# ---------------------------------------------------------------------------
# Summary — printed cleanly once all workers are done
# ---------------------------------------------------------------------------

errors  = [(f, s, log) for f, s, log in results if s.startswith('error')]
skipped = [(f, s, log) for f, s, log in results if s == 'skipped']
success = [(f, s, log) for f, s, log in results if s.startswith('success')]

print(f"\nSuccess: {len(success)}  |  Skipped: {len(skipped)}  |  Errors: {len(errors)}")
print()

# Per-file logs grouped neatly
for filename, status, log in sorted(results, key=lambda x: x[0]):
    if status == 'skipped':
        continue
    print(f"--- {filename} : {status} ---")
    if log.strip():
        for line in log.strip().splitlines():
            print(f"    {line}")
    print()

# Errors highlighted at the end
if errors:
    print("ERRORS:")
    for filename, status, log in errors:
        print(f"  {filename}: {status}")
        if log.strip():
            for line in log.strip().splitlines():
                print(f"      {line}")


DanishRainData_Outputs/30mins/All_events_5775_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5771_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5057_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5056_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5054_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5052_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5049_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5047_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5045_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5765_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5032_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_503520_precip

DanishRainData_Outputs/30mins/All_events_5414_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5409_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5411_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5412_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5407_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_540820_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5403_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5404_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_540520_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_540620_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_5397_svk_precip_minute.csv is already a file
DanishRainData_Outputs/30mins/All_events_540020_precip_minut

/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_SVK/5930_svk_precip_minute.csv
5930
/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_SVK/5930_svk_precip_minute.csv
Number of 5-min events: 1557
✅ Total matched 1557 events at 30-min resolution (start floored, end ceiled).
Trimming leading/trailing zero-precipitation periods from events...
Total events before trimming: 1501
Events skipped (all zero): 0
Events skipped (too short after trimming): 0
Events shortened: 1249
Events remaining: 1501
 Events before saving: 1501
-------------------------------
DanishRainData_Outputs/30mins/All_events_593520_precip_minute.csv is not already a file
/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData/593520_precip_minute.csv
593520
/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData/593520_precip_minute.csv
Number of 5-min events: 927
✅ Total matched 927 events at 30-min resolution (start floored, end ceiled).
Trimming leading/trailing

/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_SVK/5856_svk_precip_minute.csv
Number of 5-min events: 32
✅ Total matched 31 events at 30-min resolution (start floored, end ceiled).
Trimming leading/trailing zero-precipitation periods from events...
Total events before trimming: 30
Events skipped (all zero): 0
Events skipped (too short after trimming): 0
Events shortened: 25
Events remaining: 30
 Events before saving: 30
-------------------------------
DanishRainData_Outputs/30mins/All_events_5857_svk_precip_minute.csv is not already a file
/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_SVK/5857_svk_precip_minute.csv
5857
/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_SVK/5857_svk_precip_minute.csv
Number of 5-min events: 22
✅ Total matched 21 events at 30-min resolution (start floored, end ceiled).
Trimming leading/trailing zero-precipitation periods from events...
Total events before trimming: 21
Events skipped (all zero): 0


Number of 5-min events: 1104
✅ Total matched 1103 events at 30-min resolution (start floored, end ceiled).
Trimming leading/trailing zero-precipitation periods from events...
Total events before trimming: 1059
Events skipped (all zero): 0
Events skipped (too short after trimming): 0
Events shortened: 874
Events remaining: 1059
 Events before saving: 1059
-------------------------------
DanishRainData_Outputs/30mins/All_events_5795_svk_precip_minute.csv is not already a file
/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_SVK/5795_svk_precip_minute.csv
5795
/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_SVK/5795_svk_precip_minute.csv
Number of 5-min events: 3487
✅ Total matched 3486 events at 30-min resolution (start floored, end ceiled).
Trimming leading/trailing zero-precipitation periods from events...
Total events before trimming: 3365
Events skipped (all zero): 0
Events skipped (too short after trimming): 0
Events shortened: 2808
Events remain